In [ ]:
from flask import Flask, render_template, request
import requests
import re
import csv

app = Flask(__name__)

# Function to find emails
def get_emails(url):
    try:
        response = requests.get(url, timeout=5)
        emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", response.text)
        return list(set(emails))   # remove duplicates
    except:
        return []

# Save emails to CSV
def save_emails(url, emails):
    with open("emails.csv", "a", newline="") as f:
        writer = csv.writer(f)
        for email in emails:
            writer.writerow([url, email])

@app.route("/", methods=["GET", "POST"])
def index():

    emails = []
    url = ""

    if request.method == "POST":
        url = request.form["url"]

        if url:
            emails = get_emails(url)

            if emails:
                save_emails(url, emails)

    return render_template("index.html", emails=emails, url=url)

if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)